In [ ]:
import pandas as pd
import os
import random
import ast
import re

DATA_PATH = "student_performance_enriched.xlsx"
LIBRARY_PATH = "library_enhanced.xlsx"

df = pd.read_excel(DATA_PATH)
libraries = pd.read_excel(LIBRARY_PATH, sheet_name=None)

In [ ]:
#=============================================================
# HELPER
#=============================================================

PROFILE_METRICS = [
    "attendance_percentage",
    "weekly_self_study_hours",
    "class_participation"
]

SUBJECT_METRICS = [
    "mathematics_score",
    "science_score",
    "language_score",
    "social_studies_score"
]

SUBJECT_NAME = {
    "mathematics_score": "Mathematics",
    "science_score": "Science",
    "language_score": "Language",
    "social_studies_score": "Social Studies"
}


def classify_subject(score):
    if score >= 85:
        return "high"
    elif score >= 75:
        return "average"
    else:
        return "low"


def classify_profile(metric, score):
    metric_df = libraries["Metric Library"]
    rows = metric_df[metric_df["variable"] == metric].copy()

    # fallback kalau tidak ada min_score di library
    if metric == "attendance_percentage":
        if score >= 90:
            return "high"
        elif score >= 80:
            return "average"
        else:
            return "low"

    if metric == "weekly_self_study_hours":
        if score >= 20:
            return "high"
        elif score >= 10:
            return "average"
        else:
            return "low"

    if metric == "class_participation":
        if score >= 8:
            return "high"
        elif score >= 5:
            return "average"
        else:
            return "low"


def classify_performance_from_grade(grade):
    if grade in ["A", "B"]:
        return "high"
    elif grade == "C":
        return "average"
    else:
        return "low"

TEXT_COLUMNS = ["professional", "supportive", "strength_based", "concise"]


def choose_text(item, student_id, preferred="professional"):
    if preferred in item and pd.notna(item[preferred]):
        return item[preferred]

    available = [
        col for col in TEXT_COLUMNS
        if col in item and pd.notna(item[col])
    ]

    if not available:
        return item.get("description", "")

    random.seed(int(student_id))
    selected_col = random.choice(available)

    return item[selected_col]


def get_library_row(sheet_name, variable=None, level=None, key=None):
    lib = libraries[sheet_name]

    if key is not None:
        result = lib[lib["key"] == key]
    else:
        result = lib[
            (lib["variable"] == variable) &
            (lib["level"] == level)
        ]

    if result.empty:
        return None

    return result.iloc[0].to_dict()


def parse_learning_plan_text(value):
    """
    Gemini menghasilkan learning plan sebagai dict string.
    Function ini mengubahnya jadi text yang rapi.
    """
    if pd.isna(value):
        return ""

    if isinstance(value, dict):
        obj = value
    else:
        try:
            obj = ast.literal_eval(str(value))
        except:
            return str(value)

    goal = obj.get("goal", "")
    actions = obj.get("actions", "")
    success = obj.get("success_indicator", "")

    return f"{goal}. {actions} Success indicator: {success}"

def generate_student_facts(row):
    student_id = int(row["student_id"])

    facts = {
        "grade": row["grade"],
        "performance_level": classify_performance_from_grade(row["grade"]),

        "profile_levels": {},
        "subject_levels": {},

        # versi lama tetap dipertahankan agar tidak merusak function lain
        "profile_strengths": [],
        "subject_strengths": [],
        "development_areas": [],

        # versi baru untuk Page 2 summary
        "positive_indicators": {
            "learning_habits": [],
            "academic_performance": []
        },

        "focus_areas": {
            "learning_habits": [],
            "academic_performance": []
        },

        "learning_plans": [],
        "growth_opportunities": [],
        "balance_considerations": []
    }

    # =========================================================
    # 1. Profile / Learning Habits Interpretation
    # =========================================================
    for metric in PROFILE_METRICS:
        level = classify_profile(metric, row[metric])
        facts["profile_levels"][metric] = level

        item = get_library_row(
            "Interpretation Library",
            variable=metric,
            level=level
        )

        if item:
            text = choose_text(item, student_id)

            summary_item = {
                "label": item["title"],
                "text": text,
                "level": level,
                "metric": metric,
                "source": "Learning Habits"
            }

            if level == "high":
                facts["positive_indicators"]["learning_habits"].append(summary_item)

                facts["profile_strengths"].append({
                    "title": item["title"],
                    "text": text
                })

            else:
                facts["focus_areas"]["learning_habits"].append(summary_item)

                if level == "low":
                    facts["development_areas"].append({
                        "title": item["title"],
                        "text": text
                    })

    # =========================================================
    # 2. Subject / Academic Performance Interpretation
    # =========================================================
    for metric in SUBJECT_METRICS:
        level = classify_subject(row[metric])
        facts["subject_levels"][metric] = level

        item = get_library_row(
            "Subject Interpretation",
            variable=metric,
            level=level
        )

        if item:
            text = choose_text(item, student_id)

            summary_item = {
                "label": SUBJECT_NAME[metric],
                "text": text,
                "level": level,
                "metric": metric,
                "source": "Academic Performance",
                "title": item["title"]
            }

            if level == "high":
                facts["positive_indicators"]["academic_performance"].append(summary_item)

                facts["subject_strengths"].append({
                    "title": item["title"],
                    "text": text
                })

            else:
                facts["focus_areas"]["academic_performance"].append(summary_item)

                if level == "low":
                    facts["development_areas"].append({
                        "title": item["title"],
                        "text": text
                    })

    # =========================================================
    # 3. Learning Plan
    # Average = Suggested
    # Low     = Needs Attention
    # =========================================================
    combined_levels = {
        **facts["profile_levels"],
        **facts["subject_levels"]
    }

    for metric, level in combined_levels.items():

        if level in ["average", "low"]:
            item = get_library_row(
                "Learning Plan",
                variable=metric,
                level=level
            )

            if item:
                facts["learning_plans"].append({
                    "area": get_plan_area(metric),
                    "title": item["goal"],
                    "status": get_plan_status(level),
                    "status_class": get_plan_status_class(level),
                    "actions_html": actions_to_bullets(item["actions"]),
                    "success_indicator": item["success_indicator"],
                    "timeline": item["timeline"],
                    "level": level,
                    "metric": metric
                })

    # =========================================================
    # 4. Growth Opportunity
    # Subject score >= threshold
    # =========================================================
    growth_lib = libraries["Growth Opportunity"]

    for metric in SUBJECT_METRICS:
        score = row[metric]

        result = growth_lib[
            (growth_lib["variable"] == metric) &
            (score >= growth_lib["threshold"])
        ]

        if not result.empty:
            item = result.iloc[0].to_dict()

            facts["growth_opportunities"].append({
                "title": item["title"],
                "text": choose_text(item, student_id),
                "metric": metric,
                "score": score
            })

    # =========================================================
    # 5. Learning Observations
    # Formerly: Balance Consideration
    # =========================================================
    p = facts["performance_level"]
    attendance = facts["profile_levels"]["attendance_percentage"]
    study = facts["profile_levels"]["weekly_self_study_hours"]
    participation = facts["profile_levels"]["class_participation"]

    balance_rules = []

    if study == "high" and participation == "low":
        balance_rules.append("independent_learning_balance")

    if participation == "high" and study == "low":
        balance_rules.append("engagement_balance")

    if p == "high" and participation == "low":
        balance_rules.append("silent_achiever")

    if p == "high" and attendance == "low":
        balance_rules.append("attendance_risk")

    if study == "high" and p == "low":
        balance_rules.append("underutilized_potential")

    if study == "high" and participation == "high" and p == "low":
        balance_rules.append("learning_efficiency_balance")

    if attendance == "high" and study == "low":
        balance_rules.append("commitment_balance")

    if study == "high" and attendance == "low":
        balance_rules.append("ownership_balance")

    for key in balance_rules:
        item = get_library_row(
            "Balance Consideration",
            key=key
        )

        if item:
            facts["balance_considerations"].append({
                "title": item["title"],
                "text": choose_text(item, student_id),
                "key": key
            })

    # =========================================================
    # 6. Sort Learning Plan
    # Learning Habits first, then each subject.
    # Needs Attention first, then Suggested.
    # =========================================================
    area_priority = {
        "Learning Habits": 0,
        "Mathematics": 1,
        "Science": 2,
        "Language": 3,
        "Social Studies": 4
    }

    status_priority = {
        "Needs Attention": 0,
        "Suggested": 1
    }

    metric_priority = {
        "attendance_percentage": 0,
        "weekly_self_study_hours": 1,
        "class_participation": 2,
        "mathematics_score": 3,
        "science_score": 4,
        "language_score": 5,
        "social_studies_score": 6
    }

    facts["learning_plans"] = sorted(
        facts["learning_plans"],
        key=lambda x: (
            area_priority.get(x["area"], 99),
            status_priority.get(x["status"], 99),
            metric_priority.get(x.get("metric"), 99)
        )
    )

    return facts

In [ ]:
def bullet_html(items, empty_text="No specific notes identified."):
    if not items:
        return f"<p class='muted'>{empty_text}</p>"

    html = ""

    for item in items:
        html += f"""
        <div class="note-card">
            <div class="note-title">{item["title"]}</div>
            <div class="note-text">{item["text"]}</div>
        </div>
        """

    return html


def metric_card(label, value, level):
    color_map = {
        "high": "#16a34a",
        "average": "#f59e0b",
        "low": "#dc2626"
    }

    return f"""
    <div class="metric-card">
        <div class="metric-label">{label}</div>
        <div class="metric-value">{value}</div>
        <div class="badge" style="background:{color_map[level]}">{level.title()}</div>
    </div>
    """
def format_profile_value(metric, value):

    if metric == "attendance_percentage":
        return f"""
        <div class="metric-inline">
            <span class="metric-main">{value:.1f}%</span>
        </div>
        """

    elif metric == "weekly_self_study_hours":
        return f"""
        <div class="metric-inline">
            <span class="metric-main">{value:.1f}</span>
        </div>
        """

    elif metric == "class_participation":
        return f"""
        <div class="metric-inline">
            <span class="metric-main">{value:.1f}</span>
            <span class="metric-sub">out of 10</span>
        </div>
        """

    return f"""
    <div class="metric-inline">
        <span class="metric-main">{value:.1f}</span>
    </div>
    """


def get_plan_area(metric):

    if metric in PROFILE_METRICS:
        return "Learning Habits"

    if metric in SUBJECT_METRICS:
        return SUBJECT_NAME.get(metric, "Subject")

    return "General"


def get_plan_status(level):
    if level == "low":
        return "Needs Attention"
    elif level == "average":
        return "Suggested"
    return "Maintain"


def get_plan_status_class(level):
    if level == "low":
        return "urgent"
    elif level == "average":
        return "opportunity"
    return "maintain"


def actions_to_bullets(actions):
    if pd.isna(actions):
        return ""

    parts = str(actions).split(" | ")

    html = "<ul>"
    for action in parts:
        action = action.strip()
        if action:
            html += f"<li>{action}</li>"
    html += "</ul>"

    return html


def render_note_cards(items):
    html = ""

    for item in items:
        html += f"""
        <div class="note-card">
            <div class="note-title">{item["title"]}</div>
            <div class="note-text">{item["text"]}</div>
        </div>
        """

    return html


def render_section(title, items):
    if not items:
        return ""

    return f"""
    <div class="section">
        <h2>{title}</h2>
        {render_note_cards(items)}
    </div>
    """

def render_summary_card(title, items):
    if not items:
        return ""

    bullets = ""

    for item in items:
        bullets += f"""
        <li>
            <strong>{item["label"]}</strong><br>
            <span>{item["text"]}</span>
        </li>
        """

    return f"""
    <div class="note-card">
        <div class="note-title">{title}</div>
        <div class="note-text">
            <ul class="summary-list">
                {bullets}
            </ul>
        </div>
    </div>
    """


def render_performance_summary(facts):
    positive_html = ""
    focus_html = ""

    positive_html += render_summary_card(
        "Learning Habits",
        facts["positive_indicators"]["learning_habits"]
    )

    positive_html += render_summary_card(
        "Academic Performance",
        facts["positive_indicators"]["academic_performance"]
    )

    focus_html += render_summary_card(
        "Learning Habits",
        facts["focus_areas"]["learning_habits"]
    )

    focus_html += render_summary_card(
        "Academic Performance",
        facts["focus_areas"]["academic_performance"]
    )

    html = ""

    if positive_html:
        html += f"""
        <div class="section">
            <h2>Positive Indicators</h2>
            {positive_html}
        </div>
        """

    if focus_html:
        html += f"""
        <div class="section">
            <h2>Focus Areas</h2>
            {focus_html}
        </div>
        """

    return html

def render_learning_plan_table(plans):
    if not plans:
        return ""

    rows = ""

    for plan in plans:
        rows += f"""
        <tr>
            <td>{plan["area"]}</td>
            <td>{plan["title"]}</td>
            <td><span class="plan-status {plan["status_class"]}">{plan["status"]}</span></td>
            <td>{plan["actions_html"]}</td>
            <td>{plan["success_indicator"]}</td>
        </tr>
        """

    return f"""
    <div class="section learning-plan-section">
        <h2>Learning Plan</h2>
        <div class="plan-table-wrapper">
            <table class="plan-table">
                <thead>
                    <tr>
                        <th>Area</th>
                        <th>Plan</th>
                        <th>Status</th>
                        <th>Action</th>
                        <th>Success Indicator</th>
                    </tr>
                </thead>
                <tbody>
                    {rows}
                </tbody>
            </table>
        </div>
    </div>
    """
def generate_executive_summary(row, facts):

    student_name = row["student_name"]
    grade = row["grade"]

    grade_info = libraries["Metric Library"][
        (libraries["Metric Library"]["variable"] == "grade") &
        (libraries["Metric Library"]["level"] == grade)
    ].iloc[0].to_dict()

    # ======================================================
    # Academic Performance Summary
    # ======================================================
    high_subjects = [
        SUBJECT_NAME[m]
        for m, level in facts["subject_levels"].items()
        if level == "high"
    ]

    if len(high_subjects) == 4:

        subject_sentence = (
            "Academic performance is consistently strong across all assessed subjects."
        )

    elif len(high_subjects) == 0:

        subject_sentence = (
            "Academic performance remains balanced across subjects with opportunities for continued improvement."
        )

    elif len(high_subjects) == 1:

        subject_sentence = (
            f"Strong academic performance is demonstrated in {high_subjects[0]}."
        )

    elif len(high_subjects) == 2:

        subject_sentence = (
            f"Strong academic performance is demonstrated in "
            f"{high_subjects[0]} and {high_subjects[1]}."
        )

    else:

        last_subject = high_subjects[-1]

        subject_sentence = (
            "Strong academic performance is demonstrated in "
            + ", ".join(high_subjects[:-1])
            + f", and {last_subject}."
        )

    # ======================================================
    # Learning Habits Summary
    # ======================================================
    habit_labels = {
        "attendance_percentage": "attendance consistency",
        "weekly_self_study_hours": "weekly study routines",
        "class_participation": "classroom participation"
    }

    focus_habits = [
        habit_labels[m]
        for m, level in facts["profile_levels"].items()
        if level in ["average", "low"]
    ]

    if len(focus_habits) == 0:

        habit_sentence = (
            "Learning habits provide a strong foundation for continued academic success."
        )

    elif len(focus_habits) == 1:

        habit_sentence = (
            f"Further development in {focus_habits[0]} would strengthen overall learning effectiveness."
        )

    elif len(focus_habits) == 2:

        habit_sentence = (
            f"Further development in {focus_habits[0]} and {focus_habits[1]} "
            "would strengthen overall learning effectiveness."
        )

    else:

        last = focus_habits[-1]

        habit_sentence = (
            "Further development in "
            + ", ".join(focus_habits[:-1])
            + f", and {last} would strengthen overall learning effectiveness."
        )

    # ======================================================
    # Learning Plan Summary
    # ======================================================
    plan_count = len(facts["learning_plans"])

    if plan_count == 0:

        plan_sentence = (
            "No immediate learning plan is recommended based on the current assessment profile."
        )

    elif plan_count == 1:

        plan_sentence = (
            "One targeted learning plan is recommended to support continued development."
        )

    else:

        plan_sentence = (
            f"{plan_count} targeted learning plans are recommended to support continued development."
        )

    # ======================================================
    # Final Executive Summary
    # ======================================================
    return (
        f"{student_name} achieved a final grade of "
        f"{grade}, categorized as {grade_info['label']}. "
        f"{subject_sentence} "
        f"{habit_sentence} "
        f"{plan_sentence}"
    )

def render_report_header(title, portfolio=True):

    if portfolio:
        return f"""
        <div class="header">

            <h1>{title}</h1>

            <div class="subtitle">
                End-to-end reporting pipeline powered by structured analytics,
                knowledge libraries, and LLM-assisted interpretation.
            </div>

            <div class="portfolio">
                Portfolio Project • Anandya Devan Alfarizky
            </div>

            <div class="portfolio-contact">
                anandyadevan@gmail.com • linkedin.com/in/anandyadevan
            </div>

        </div>
        """

    else:
        return f"""
        <div class="header">
            <h1>{title}</h1>
        </div>
        """


def render_report_footer(page_number):
    return f"""
    <div class="footer">
        <div class="footer-left">
            Page {page_number}
        </div>

        <div class="footer-right">
            <strong>Portfolio Project by Anandya Devan Alfarizky</strong><br>
            anandyadevan@gmail.com • linkedin.com/in/anandyadevan
        </div>
    </div>
    """

def generate_student_html(row):
    facts = generate_student_facts(row)

    student_name = row["student_name"]
    student_id = int(row["student_id"])
    grade = row["grade"]

    grade_info = libraries["Metric Library"][
        (libraries["Metric Library"]["variable"] == "grade") &
        (libraries["Metric Library"]["level"] == grade)
    ].iloc[0].to_dict()

    # =========================================================
    # Learning Profile Cards
    # =========================================================
    profile_cards = ""

    for metric, label in [
        ("attendance_percentage", "Attendance"),
        ("weekly_self_study_hours", "Weekly Study Hours"),
        ("class_participation", "Class Participation")
    ]:
        profile_cards += metric_card(
            label,
            format_profile_value(metric, row[metric]),
            facts["profile_levels"][metric]
        )

    # =========================================================
    # Subject Performance Rows
    # =========================================================
    subject_rows = ""

    for metric in SUBJECT_METRICS:
        level = facts["subject_levels"][metric]
        score = row[metric]

        subject_rows += f"""
        <tr>
            <td>{SUBJECT_NAME[metric]}</td>
            <td>{score:.1f}</td>
            <td><span class="small-badge {level}">{level.title()}</span></td>
        </tr>
        """

    # =========================================================
    # Executive Summary
    # =========================================================
    executive_summary = generate_executive_summary(row, facts)
    
    # =========================================================
    # Page 2 Dynamic Sections
    # Only render sections when data exists
    # =========================================================
    page2_sections = ""

    page2_sections += render_performance_summary(facts)

    page2_sections += render_learning_plan_table(
        facts["learning_plans"]
    )

    page2_sections += render_section(
        "Growth Opportunities",
        facts["growth_opportunities"]
    )

    page2_sections += render_section(
        "Learning Observations",
        facts["balance_considerations"]
    )

    page1_header = render_report_header("Student Performance Report",portfolio=True)
    page2_header = render_report_header("Learning & Development Summary",portfolio=False)

    page1_footer = render_report_footer(1)
    page2_footer = render_report_footer(2)
    
    html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>Student Performance Report</title>

<style>
@page {{
    size: A4;
    margin: 0;
}}

* {{
    box-sizing: border-box;
}}

body {{
    margin: 0;
    padding: 0;
    background: #e5e7eb;
    font-family: Arial, sans-serif;
    color: #111827;
}}

.page {{
    width: 210mm;
    min-height: 297mm;
    margin: 0 auto;
    background: white;
    padding: 18mm;
    page-break-after: always;
}}

.header {{
    border-bottom: 3px solid #111827;
    padding-bottom: 14px;
    margin-bottom: 22px;
}}

.header h1 {{
    margin: 0;
    font-size: 28px;
}}

.subtitle {{
    color:#6b7280;
    margin-top:6px;
    font-size:11.5px;
    line-height:1.5;
    max-width:720px;
}}

.portfolio {{
    margin-top:8px;
    font-size:12px;
    font-weight:600;
    color:#111827;
}}

.portfolio-contact {{
    margin-top:3px;
    font-size:11px;
    color:#6b7280;
}}

.footer {{
    margin-top:24px;
    border-top:1px solid #e5e7eb;
    padding-top:10px;
    display:flex;
    justify-content:space-between;
    align-items:flex-start;
    font-size:10px;
    color:#6b7280;
}}

.footer-left {{
    font-weight:600;
}}

.footer-right {{
    text-align:right;
    line-height:1.5;
}}

.profile-card {{
    background: #f3f4f6;
    border-radius: 16px;
    padding: 22px;
    margin-bottom: 22px;
}}

.student-name {{
    font-size: 30px;
    font-weight: 700;
}}

.student-id {{
    font-size: 13px;
    color: #6b7280;
    margin-top: 6px;
}}

.grade-row {{
    margin-top: 18px;
    display: flex;
    align-items: center;
    gap: 18px;
}}

.grade {{
    font-size: 52px;
    font-weight: 800;
    color: {grade_info["color"]};
}}

.grade-label {{
    font-size: 18px;
    font-weight: 700;
}}

.grade-desc {{
    color: #4b5563;
    font-size: 13px;
    margin-top: 4px;
}}

.section {{
    margin-top: 22px;
}}

.section h2 {{
    font-size: 18px;
    margin-bottom: 10px;
    border-bottom: 1px solid #d1d5db;
    padding-bottom: 8px;
}}

.metric-grid {{
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 12px;
}}

.metric-card {{
    background: #f9fafb;
    border: 1px solid #e5e7eb;
    border-radius: 14px;
    padding: 14px;
}}

.metric-label {{
    color: #6b7280;
    font-size: 12px;
}}

.metric-value {{
    margin-top: 6px;
}}

.metric-inline {{
    display: flex;
    align-items: center;
    gap: 6px;
}}

.metric-main {{
    font-size: 23px;
    font-weight: 700;
    line-height: 1;
}}

.metric-sub {{
    font-size: 12px;
    color: #9ca3af;
    font-weight: 400;
    line-height: 2px;
}}

.badge {{
    display: inline-block;
    color: white;
    padding: 4px 9px;
    border-radius: 999px;
    font-size: 11px;
    margin-top: 8px;
}}

.subject-table {{
    width: 100%;
    border-collapse: collapse;
    margin-top: 8px;
}}

.subject-table th,
.subject-table td {{
    border-bottom: 1px solid #e5e7eb;
    text-align: left;
    padding: 10px;
    font-size: 13px;
}}

.subject-table th {{
    background: #f9fafb;
}}

.small-badge {{
    color: white;
    padding: 4px 8px;
    border-radius: 999px;
    font-size: 11px;
}}

.small-badge.high {{
    background: #16a34a;
}}

.small-badge.average {{
    background: #f59e0b;
}}

.small-badge.low {{
    background: #dc2626;
}}

.summary-box {{
    background: #f9fafb;
    border-radius: 14px;
    padding: 14px;
    font-size: 13px;
    line-height: 1.6;
}}

.note-card {{
    border: 1px solid #e5e7eb;
    border-left: 5px solid #111827;
    border-radius: 12px;
    padding: 12px 14px;
    margin-bottom: 10px;
    background: #ffffff;
}}

.note-title {{
    font-weight: 700;
    margin-bottom: 5px;
}}

.note-text {{
    font-size: 13px;
    color: #374151;
    line-height: 1.5;
}}

.summary-list {{
    margin: 0;
    padding-left: 18px;
}}

.summary-list li {{
    margin-bottom: 8px;
    line-height: 1.45;
}}

.summary-list li:last-child {{
    margin-bottom: 0;
}}

.summary-list span {{
    color: #374151;
}}

.plan-table {{
    width: 100%;
    border-collapse: collapse;
    margin-top: 0;
}}

.plan-table-wrapper {{
    border: 1px solid #e5e7eb;
    border-radius: 14px;
    overflow: hidden;
    background: white;
    margin-top: 8px;
}}

.plan-table th:first-child {{
    border-top-left-radius:8px;
}}

.plan-table th:last-child {{
    border-top-right-radius:8px;
}}

.plan-table th,
.plan-table td {{
    border-bottom: 1px solid #e5e7eb;
    text-align: left;
    vertical-align: top;
    padding:14px 12px;
    font-size: 11.5px;
}}

.plan-table th {{
    background: #111827;
    color: white;
    font-weight: 600;
    text-align: left;
    padding: 10px;
    font-size: 12px;
}}

.plan-table tbody tr:nth-child(even) {{
    background: #f3f4f6;
}}

.plan-table ul {{
    margin: 0;
    padding-left: 16px;
}}

.plan-table li {{
    margin-bottom: 2px;
    line-height:1.35;
}}

.plan-status {{
    display: inline-block;
    color: white;
    padding: 6px 12px;
    border-radius: 999px;
    font-size: 11px;
    font-weight:600;
    white-space: nowrap;
}}

.plan-status.urgent {{
    background: #dc2626;
}}

.plan-status.opportunity {{
    background: #f59e0b;
}}

.plan-status.maintain {{
    background: #16a34a;
}}

@media print {{
    body {{
        background: white;
    }}

    .page {{
        margin: 0;
        width: 210mm;
        min-height: 297mm;
        page-break-after: always;
    }}
}}
</style>
</head>

<body>

<div class="page">

    {page1_header}

    <div class="profile-card">
        <div class="student-name">{student_name}</div>
        <div class="student-id">Student ID: {student_id}</div>

        <div class="grade-row">
            <div class="grade">{grade}</div>
            <div>
                <div class="grade-label">{grade_info["label"]}</div>
                <div class="grade-desc">{grade_info["description"]}</div>
            </div>
        </div>
    </div>

    <div class="section">
        <h2>Executive Summary</h2>
        <div class="summary-box">{executive_summary}</div>
    </div>

    <div class="section">
        <h2>Learning Habits</h2>
        <div class="metric-grid">
            {profile_cards}
        </div>
    </div>

    <div class="section">
        <h2>Subject Performance</h2>
        <table class="subject-table">
            <thead>
                <tr>
                    <th>Subject</th>
                    <th>Score</th>
                    <th>Level</th>
                </tr>
            </thead>
            <tbody>
                {subject_rows}
            </tbody>
        </table>
    </div>

    {page1_footer}

</div>

</div>

<div class="page">

    {page2_header}

    {page2_sections}

    {page2_footer}

</div>

</body>
</html>
"""

    return html

In [ ]:
import os
import re
import pandas as pd

DATA_PATH = "student_performance_enriched.xlsx"
LIBRARY_PATH = "library_enhanced.xlsx"
OUTPUT_DIR = "outputs_html"
MAX_REPORTS = 1000

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_excel(DATA_PATH)
libraries = pd.read_excel(LIBRARY_PATH, sheet_name=None)


def clean_filename(value):
    """
    Remove characters that are not allowed in Windows file names.
    """
    return re.sub(
        r'[<>:"/\\|?*]',
        "",
        str(value)
    ).strip()


def build_report_filename(student):
    student_id = int(student["student_id"])
    student_name = clean_filename(student["student_name"])

    return (
        f"Report Automation by anandyadevan - "
        f"ID {student_id} - {student_name}.html"
    )


generated_files = []

for _, student in df.head(MAX_REPORTS).iterrows():

    html = generate_student_html(student)

    filename = build_report_filename(student)

    html_path = os.path.join(
        OUTPUT_DIR,
        filename
    )

    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)

    generated_files.append(html_path)


print(f"Generated {len(generated_files)} HTML reports.")
print(f"Output folder: {OUTPUT_DIR}")